# 3.0.0-Normalize and dimensionlity reduction

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
options(warn = -1, verbose = FALSE)
suppressMessages(suppressWarnings({
library(Seurat)
library(Signac)
library(readr)
library(tidyverse)
library(ggplot2)
library(GenomicRanges)
library(scran)
library(scater)
}))

## Parameters

In [ ]:
args <- commandArgs(TRUE)

patient <- args[2]
subproj <- args[3]

In [ ]:
source(here::here(glue::glue("{subproj}/multiome_analysis/misc/paths.R")))
source(here::here(glue::glue("{subproj}/multiome_analysis/utils/variables.R")))

"{clust}/{plt_dir}" %>%
  glue::glue() %>%
  here::here() %>%
  dir.create(path = .,
           showWarnings = FALSE,
           recursive = TRUE)

"{clust}/{robj_dir}" %>%
  glue::glue() %>%
  here::here() %>%
  dir.create(path = .,
           showWarnings = FALSE,
           recursive = TRUE)

set.seed(0)

In [ ]:
var_name <- paste0("comp_", patient)
dcomp <- get(var_name)
dcomp

## Load data

In [ ]:
# clean seurat object
seurat_obj <- readRDS(here::here(glue::glue("{qc}/{robj_dir}/2.3_{patient}.rds")))

In [ ]:
# variables
cat_vars <-c("is_cell", "Phase")
con_vars <- c("nCount_RNA", "nFeature_RNA", "nCount_ATAC",
              "nFeature_ATAC", "pct_mt", "pct_ribosomal","PTPRC",
              "doublet_score", "doublet_score_atac", "doublet_enrichment_atac")
vars <- c(cat_vars, con_vars)

## RNA

In this step, data is normalized as the feature expression measurements for each cell by the total expression, then multiplies this by a scale factor (10,000 by default), and log-transforms the result.

In [ ]:
DefaultAssay(seurat_obj) <- "RNA"
seurat_obj <- NormalizeData(
  seurat_obj,
  normalization.method = "LogNormalize",
  scale.factor = 1e4
)

### Compute Cell-Cycle scores

In [ ]:
# A list of cell cycle markers, from Tirosh et al, 2015, is loaded with Seurat.  We can
# segregate this list into markers of G2/M phase and markers of S phase
s.genes <- cc.genes$s.genes
g2m.genes <- cc.genes$g2m.genes
seurat_obj <- CellCycleScoring(seurat_obj, s.features = s.genes, g2m.features = g2m.genes, set.ident = TRUE)

### Dimentionality reduction

In [ ]:
# compute the variance and mean of the log-expression values 
sce <- as.SingleCellExperiment(seurat_obj)
gene_var <- modelGeneVar(sce)

tops <- gene_var %>% 
    as.data.frame() %>% 
    arrange(desc(total)) %>% 
    head(n=20)

plt <- gene_var %>% 
  as.data.frame() %>% 
  ggplot(aes(mean, total)) +
  geom_point() +
  geom_line(aes(y = tech), colour = "dodgerblue", size = 1) +
  labs(x = "Mean of log-expression", y = "Variance of log-expression")+
  geom_text(data=tops, aes(mean,total,label=rownames(tops)))

hvgs <- getTopHVGs(gene_var,fdr.threshold = 0.05)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5)
plt

In [ ]:
length(hvgs)

In [ ]:
VariableFeatures(seurat_obj) <- hvgs

seurat_obj <- seurat_obj %>%
  ScaleData() %>% 
  RunPCA() 

In [ ]:
ElbowPlot(seurat_obj, n=50)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
FeaturePlot(object = seurat_obj, reduction = "pca",
        features = c("doublet_score", "PTPRC"), order=T)

In [ ]:
seurat_obj <- RunUMAP(
  seurat_obj,
  dims = dcomp$rna,
  reduction = "pca",
  reduction.key = "rnaUMAP_",
   assay.use = "RNA",
)

In [ ]:
# compute plots
list_plots <- lapply(vars, function(var){
  if (var %in% cat_vars) {
    if (var == "is_cell") {
      p <- DimPlot(seurat_obj, reduction = "umap", group.by=var) +
        scale_color_manual(breaks = c(0, 1), values=c("#56B4E9", "#E69F00"))
    } else {
      p <- DimPlot(seurat_obj, reduction = "umap", group.by=var)
    }
  } else {
    p <- FeaturePlot(seurat_obj, reduction = "umap", feature=var, order = FALSE)
  }
  return(p)
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
# show plots
cp <- cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol = 5,
                   nrow = 2)
cp

## ATAC

In [ ]:
DefaultAssay(seurat_obj) <- "ATAC" # for individual analysis
seurat_obj <- RunTFIDF(seurat_obj)

In [ ]:
seurat_obj <- FindTopFeatures(seurat_obj, min.cutoff = "q75")
seurat_obj <- RunSVD(seurat_obj, reduction.name = "lsi.atac")

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
DepthCor(seurat_obj, reduction = "lsi.atac", n=40)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5)
ElbowPlot(seurat_obj, n=20, reduction="lsi.atac")

In [ ]:
seurat_obj <- RunUMAP(
  seurat_obj,
  dims = dcomp$atac,
  reduction = "lsi.atac",
  reduction.name = "umap.atac",
  reduction.key = "atacUMAP_"
)

In [ ]:
list_plots <- lapply(vars, function(var){
  if (var %in% cat_vars) {
    if (var == "is_cell") {
      p <- DimPlot(seurat_obj, reduction = "umap.atac", group.by=var) +
        scale_color_manual(breaks = c(0, 1), values=c("#56B4E9", "#E69F00"))
    } else {
      p <- DimPlot(seurat_obj, reduction = "umap.atac", group.by=var)
    }
  } else {
    p <- FeaturePlot(seurat_obj, reduction = "umap.atac", feature=var, order = TRUE)
  }
  return(p)
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
cp <- cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol = 5,
                   nrow = 2)
cp

In [ ]:
comp_variances <- as.data.frame(seurat_obj@reductions$lsi.atac@feature.loadings)
comp_variances$peak <- row.names(comp_variances)

head(comp_variances %>% 
 arrange(desc(LSI_2)))

Overlap gene anotations with peaks with most influence on LSI 2

In [ ]:
# get the first 50 peaks
var_peaks <- comp_variances %>% 
 arrange(desc(LSI_2)) %>% pull(peak)
var_peaks <- var_peaks[1:50]
# convert it into a GRanges object
var_peaks <- sub('-', ':', var_peaks)
var_peaks <- GRanges(as.data.frame((GRangesList(var_peaks))))

In [ ]:
# get gene coordinates from Biomart
mart_gb <- readr::read_tsv(here::here(glue::glue("external_data/mart_export_GRCh38-p13.txt")))
colnames(mart_gb) <- c("end", "start", "chrom", "hugo_symbol", "canonical", "ensembl_id")

# keep the ones in standard chromosomes
standard_chroms <- c(as.character(seq(1, 22, by=1)))
standard_chroms <- c(standard_chroms, c("X","Y"))
mart_gb <- dplyr::filter(mart_gb, canonical == 1) %>% dplyr::filter(chrom %in% standard_chroms)

# convert it into a Grange object
gene_coords <- GRanges(seqnames = paste0("chr", mart_gb$chrom),
                    ranges = IRanges(start = mart_gb$start, end = mart_gb$end),
                    names = mart_gb$hugo_symbol)

# get overlapping gene peaks
overlapping_genes<- gene_coords[queryHits(findOverlaps(gene_coords,var_peaks)),]
overlapping_genes$names

## Joint

In [ ]:
seurat_obj <- FindMultiModalNeighbors(
 seurat_obj,
 reduction.list = list("pca", "lsi.atac"),
 dims.list = list(dcomp$rna, dcomp$atac)
)

In [ ]:
seurat_obj <- RunUMAP(
   seurat_obj,
   nn.name = "weighted.nn",
   reduction.name = "wnn.umap",
   reduction.key = "wnnUMAP_"
)

In [ ]:
# compute plots
list_plots <- lapply(vars, function(var){
  if (var %in% cat_vars) {
    if (var == "is_cell") {
      p <- DimPlot(seurat_obj, reduction = "wnn.umap", group.by=var) +
        scale_color_manual(breaks = c(0, 1), values=c("#56B4E9", "#E69F00"))
    } else {
      p <- DimPlot(seurat_obj, reduction = "wnn.umap", group.by=var)
    }
  } else {
    p <- FeaturePlot(seurat_obj, reduction = "wnn.umap", feature=var, order = TRUE)
  }
  return(p)
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
cp <- cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol = 5,
                   nrow = 2)
cp

## Save

In [ ]:
saveRDS(seurat_obj, here::here(glue::glue("{clust}/{robj_dir}/3.0.0-lognorm_{patient}.rds")))

## Session

In [ ]:
sessionInfo()